# Day 08 RAG Pipeline Demo: Task 1 -> Task 10

Notebook này là một mẫu chạy từng bước để xem luồng dữ liệu từ raw documents/news -> markdown -> chunks -> retrieval -> generation có citation.

Mặc định các bước tốn API hoặc ghi index được tắt. Bật các flag ở cell Setup nếu muốn chạy đầy đủ.

In [86]:
from pathlib import Path
from datetime import datetime
from html import unescape
from html.parser import HTMLParser
import json
import math
import os
import re
import time

import requests
from dotenv import load_dotenv

# Notebook tự chứa: không import từ src/task*.py.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
load_dotenv(PROJECT_ROOT / ".env")

LANDING_DIR = PROJECT_ROOT / "data" / "landing"
LEGAL_DIR = LANDING_DIR / "legal"
NEWS_DIR = LANDING_DIR / "news"
STANDARDIZED_DIR = PROJECT_ROOT / "data" / "standardized"
PAGEINDEX_DOC_IDS_PATH = PROJECT_ROOT / "data" / "pageindex_doc_ids.json"

# Flags: bật True nếu muốn gọi mạng/API hoặc ghi index.
RUN_TASK1_DOWNLOAD = False
RUN_TASK2_CRAWL = False
RUN_TASK3_CONVERT = False
RUN_TASK4_INDEX = False       # Gọi OpenAI embedding + ghi Weaviate
RUN_SEMANTIC_SEARCH = True    # Có fallback local nếu OpenAI/Weaviate lỗi
RUN_PAGEINDEX_UPLOAD = False  # Gọi PageIndex API
RUN_GENERATION = False        # True để gọi LLM; False dùng extractive fallback
RUN_TASK11_RAGAS = False      # True để chạy RAGAS judge; tốn OpenAI calls
RAGAS_EVAL_LIMIT = 3          # Số golden cases khi thử RAGAS trong notebook

print("Project root:", PROJECT_ROOT)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("PAGEINDEX_API_KEY set:", bool(os.getenv("PAGEINDEX_API_KEY")))
print("Standardized dir:", STANDARDIZED_DIR)


Project root: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2
OPENAI_API_KEY set: True
PAGEINDEX_API_KEY set: True
Standardized dir: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/standardized


## Task 1 - Thu thập văn bản pháp luật

Mục tiêu: tải hoặc kiểm tra các file PDF/DOCX trong `data/landing/legal/`.

In [87]:
# Task 1 - Thu thập văn bản pháp luật
# Mục tiêu: tải PDF luật vào data/landing/legal.

LEGAL_URLS = {
    "Luat_Phong_Chong_Ma_Tuy_2021.pdf": "https://datafiles.chinhphu.vn/cpp/files/vbpq/2022/01/73luat.pdf",
    "Luat_120_2025.pdf": "https://cdn.thuviennhadat.vn/upload/hinh-anh-bai-viet/DTTM/2026/luat-phong-chong-ma-tuy-2025.pdf",
    "Luat_phong_chong_ma_tuy_Nghi_Duong_Hai_Phong_2025.pdf": "https://cdn.haiphong.gov.vn/gov-hpg/6836/tintuc/2026/3/kh-trien-khai-thi-hanh-luat-ma-tuy-2025-ubnd.signed639082147594116577.pdf",
}

LEGAL_DIR.mkdir(parents=True, exist_ok=True)


def download_file(url: str, output_path: Path, overwrite: bool = False):
    if output_path.exists() and not overwrite:
        print("Skip existing:", output_path.name)
        return output_path

    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=(10, 60)) as response:
        response.raise_for_status()
        tmp_path = output_path.with_suffix(output_path.suffix + ".part")
        with tmp_path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 64):
                if chunk:
                    f.write(chunk)
        tmp_path.replace(output_path)

    if output_path.stat().st_size < 1024:
        raise ValueError(f"File quá nhỏ, có thể tải lỗi: {output_path}")
    return output_path


def download_legal_docs(overwrite: bool = False):
    for filename, url in LEGAL_URLS.items():
        print("Downloading:", filename)
        try:
            download_file(url, LEGAL_DIR / filename, overwrite=overwrite)
        except Exception as exc:
            print("  Error:", exc)

if RUN_TASK1_DOWNLOAD:
    download_legal_docs(overwrite=False)

legal_files = sorted(p for p in LEGAL_DIR.iterdir() if p.suffix.lower() in {".pdf", ".doc", ".docx"})
print(f"Legal files: {len(legal_files)}")
for p in legal_files:
    print(f"- {p.name}: {p.stat().st_size:,} bytes")


Legal files: 4
- Luat_120_2025.pdf: 10,366,370 bytes
- Luat_Phong_Chong_Ma_Tuy_2021.pdf: 537,045 bytes
- Luat_phong_chong_ma_tuy_Nghi_Duong_Hai_Phong_2025.pdf: 1,026,002 bytes
- Nghi_Dinh_28_2026_ND_CP.pdf: 1,471,070 bytes


## Task 2 - Crawl bài báo

Mục tiêu: crawl tối thiểu 5 bài báo và lưu JSON trong `data/landing/news/`.

In [88]:
# Task 2 - Crawl bài báo bằng Crawl4AI
# Mục tiêu: lưu JSON gồm url, title, date_crawled, content_markdown vào data/landing/news.

import asyncio

NEWS_DIR.mkdir(parents=True, exist_ok=True)
ARTICLE_URLS = [
    "https://thanhnien.vn/ca-si-long-nhat-bi-bat-showbiz-viet-lien-tiep-chan-dong-vi-ma-tuy-18526052013032001.htm",
    "https://baochinhphu.vn/khoi-to-bat-tam-giam-ca-si-long-nhat-son-ngoc-minh-vi-to-chuc-su-dung-ma-tuy-102260520125739676.htm",
    "https://vov.vn/giai-tri/chua-day-1-thang-3-nghe-si-viet-bi-khoi-to-vi-lien-quan-ma-tuy-gay-chan-dong-post1293496.vov",
    "https://danviet.vn/nhuc-nhoi-loat-nghe-si-vuong-lao-ly-vi-ma-tuy-khong-chi-la-sa-nga-ma-con-la-su-ton-thuong-doi-voi-niem-tin-cong-chung-d1428424.html",
    "https://vietnamnet.vn/loat-ca-si-dinh-chat-cam-ma-tuy-pha-huy-nao-bo-nguoi-tre-ra-sao-2518285.html",
]


def get_crawl4ai_markdown(result) -> str:
    """Crawl4AI có phiên bản trả markdown là str, có phiên bản trả object."""
    markdown = getattr(result, "markdown", "")
    if isinstance(markdown, str):
        return markdown.strip()

    for attr in ("fit_markdown", "raw_markdown", "markdown"):
        value = getattr(markdown, attr, None)
        if isinstance(value, str) and value.strip():
            return value.strip()

    return str(markdown).strip()


async def crawl_article(url: str) -> dict:
    from crawl4ai import AsyncWebCrawler

    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(url=url)

    metadata = getattr(result, "metadata", {}) or {}
    content_markdown = get_crawl4ai_markdown(result)

    return {
        "url": url,
        "title": metadata.get("title") or "Unknown",
        "date_crawled": datetime.now().isoformat(),
        "date_published": metadata.get("published_time", ""),
        "crawler": "crawl4ai",
        "content": content_markdown,
        "content_markdown": content_markdown,
    }


async def crawl_all_news():
    for i, url in enumerate(ARTICLE_URLS, 1):
        print(f"[{i}/{len(ARTICLE_URLS)}] {url}")
        article = await crawl_article(url)

        output_path = NEWS_DIR / f"article_{i:02d}.json"
        output_path.write_text(
            json.dumps(article, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print("  Saved:", output_path.name, "chars=", len(article["content_markdown"]))


if RUN_TASK2_CRAWL:
    await crawl_all_news()

article_jsons = sorted(NEWS_DIR.glob("article_*.json"))
print(f"News JSON files: {len(article_jsons)}")
for p in article_jsons[:5]:
    data = json.loads(p.read_text(encoding="utf-8"))
    print(f"- {p.name}: {data.get('title', 'Unknown')} | chars={len(data.get('content_markdown', ''))}")


News JSON files: 5
- article_01.json: Ca sĩ Long Nhật bị bắt, showbiz Việt liên tiếp chấn động vì ma túy | chars=13654
- article_02.json: Khởi tố, bắt tạm giam ca sĩ Long Nhật, Sơn Ngọc Minh vì tổ chức sử dụng ma túy | chars=12206
- article_03.json: Chưa đầy 1 tháng, 3 nghệ sĩ Việt bị khởi tố vì liên quan ma túy gây chấn động | chars=54638
- article_04.json: Nhức nhối loạt nghệ sĩ vướng lao lý vì ma túy: Không chỉ là sa ngã mà còn là sự tổn thương đối với niềm tin công chúng | chars=91752
- article_05.json: Loạt ca sĩ 'dính' chất cấm, ma túy phá hủy não bộ người trẻ ra sao? | chars=26853


## Task 3 - Convert sang Markdown

Mục tiêu: chuyển legal/news sang markdown trong `data/standardized/`.

In [89]:
# Task 3 - Convert sang Markdown
# Legal PDF/DOCX dùng MarkItDown; news JSON chỉ cần ghép metadata + content_markdown.

OUTPUT_DIR = STANDARDIZED_DIR


def convert_legal_docs():
    output_dir = OUTPUT_DIR / "legal"
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        from markitdown import MarkItDown
    except ImportError:
        print("MarkItDown chưa được cài. Bỏ qua legal conversion.")
        return

    md = MarkItDown()
    for filepath in sorted(LEGAL_DIR.iterdir()):
        if filepath.suffix.lower() not in {".pdf", ".docx", ".doc"}:
            continue
        print("Converting legal:", filepath.name)
        try:
            result = md.convert(str(filepath))
            (output_dir / f"{filepath.stem}.md").write_text(result.text_content, encoding="utf-8")
        except Exception as exc:
            print("  Skip vì lỗi convert:", exc)


def convert_news_articles():
    output_dir = OUTPUT_DIR / "news"
    output_dir.mkdir(parents=True, exist_ok=True)

    for filepath in sorted(NEWS_DIR.glob("*.json")):
        data = json.loads(filepath.read_text(encoding="utf-8"))
        header = f"# {data.get('title', 'Unknown')}\n\n"
        header += f"**Source:** {data.get('url', 'N/A')}\n"
        header += f"**Crawled:** {data.get('date_crawled', 'N/A')}\n\n---\n\n"
        content = header + data.get("content_markdown", "")
        (output_dir / f"{filepath.stem}.md").write_text(content, encoding="utf-8")
        print("Converted news:", filepath.name)


def convert_all():
    convert_legal_docs()
    convert_news_articles()

if RUN_TASK3_CONVERT:
    convert_all()

md_files = sorted(STANDARDIZED_DIR.rglob("*.md"))
print(f"Markdown files: {len(md_files)}")
for p in md_files:
    print(f"- {p.relative_to(STANDARDIZED_DIR)}: {p.stat().st_size:,} bytes")

if md_files:
    print("\nPreview:\n", md_files[0].read_text(encoding="utf-8")[:800])


Markdown files: 9
- legal/Luat_120_2025.md: 0 bytes
- legal/Luat_Phong_Chong_Ma_Tuy_2021.md: 79,009 bytes
- legal/Luat_phong_chong_ma_tuy_Nghi_Duong_Hai_Phong_2025.md: 8,367 bytes
- legal/Nghi_Dinh_28_2026_ND_CP.md: 0 bytes
- news/article_01.md: 16,021 bytes
- news/article_02.md: 14,246 bytes
- news/article_03.md: 61,734 bytes
- news/article_04.md: 107,281 bytes
- news/article_05.md: 30,374 bytes

Preview:
 


## Task 4 - Load Documents

Mục tiêu: đọc markdown files.

In [90]:
# Task 4a - Load Documents
# Đọc markdown trong data/standardized và gắn metadata nguồn.


def load_documents() -> list[dict]:
    documents = []
    for md_file in sorted(STANDARDIZED_DIR.rglob("*.md")):
        content = md_file.read_text(encoding="utf-8").strip()
        if not content:
            continue
        relative_path = md_file.relative_to(STANDARDIZED_DIR)
        documents.append({
            "content": content,
            "metadata": {
                "source": md_file.name,
                "path": str(relative_path),
                "type": relative_path.parts[0] if relative_path.parts else "unknown",
            },
        })
    return documents


docs = load_documents()
print(f"Documents: {len(docs)}")
if docs:
    print(docs[0]["content"][:300])
    print(docs[0]["metadata"])


Documents: 7
4

CÔNG BÁO/Số 567 + 568/Ngày 30-4-2021

QUỐC HỘI

Luật số: 73/2021/QH14

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc

LUẬT
PHÒNG, CHỐNG MA TÚY

Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;

Quốc hội ban hành Luật Phòng, chống ma túy.

Chương I
NHỮNG QUY ĐỊNH CHUNG

Đ
{'source': 'Luat_Phong_Chong_Ma_Tuy_2021.md', 'path': 'legal/Luat_Phong_Chong_Ma_Tuy_2021.md', 'type': 'legal'}


## Task 4 - Chunking

Mục tiêu: chia documents thành chunks.

In [91]:
# Task 4b - Chunking
# Recursive splitter: ưu tiên tách theo đoạn/câu, overlap để giữ ngữ cảnh.

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
CHUNKING_METHOD = "recursive"


def simple_split_text(text: str) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - CHUNK_OVERLAP
    return [chunk for chunk in chunks if chunk]


def chunk_documents(documents: list[dict]) -> list[dict]:
    try:
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        split_text = splitter.split_text
    except ImportError:
        split_text = simple_split_text

    chunks = []
    for doc in documents:
        for i, chunk_text in enumerate(split_text(doc["content"])):
            chunks.append({
                "content": chunk_text,
                "metadata": {
                    **doc["metadata"],
                    "chunk_index": i,
                    "chunking_method": CHUNKING_METHOD,
                },
            })
    return chunks


chunks = chunk_documents(docs)
print(f"Chunks: {len(chunks)}")
print(f"Chunk config: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
if chunks:
    print(chunks[0]["content"][:300])
    print(chunks[0]["metadata"])


Chunks: 709
Chunk config: size=500, overlap=50
4

CÔNG BÁO/Số 567 + 568/Ngày 30-4-2021

QUỐC HỘI

Luật số: 73/2021/QH14

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc

LUẬT
PHÒNG, CHỐNG MA TÚY

Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;

Quốc hội ban hành Luật Phòng, chống ma túy.

Chương I
NHỮNG QUY ĐỊNH CHUNG

Đ
{'source': 'Luat_Phong_Chong_Ma_Tuy_2021.md', 'path': 'legal/Luat_Phong_Chong_Ma_Tuy_2021.md', 'type': 'legal', 'chunk_index': 0, 'chunking_method': 'recursive'}


## Task 4 - Embedding và Indexing

Mục tiêu: embed chunks bằng OpenAI và index vào Weaviate Docker.

In [92]:
# Task 4c - Embedding và Indexing vào Weaviate
# Bước này gọi OpenAI embedding và ghi vector store, nên chỉ chạy khi RUN_TASK4_INDEX=True.

EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
OPENAI_EMBEDDING_DIM = os.getenv("OPENAI_EMBEDDING_DIM", "")
EMBEDDING_DIM = int(OPENAI_EMBEDDING_DIM or 1536)
EMBEDDING_BATCH_SIZE = int(os.getenv("OPENAI_EMBEDDING_BATCH_SIZE", "64"))
WEAVIATE_COLLECTION = os.getenv("WEAVIATE_COLLECTION", "DrugLawDocs")
WEAVIATE_HOST = os.getenv("WEAVIATE_HOST", "localhost")
WEAVIATE_PORT = int(os.getenv("WEAVIATE_PORT", "8080"))
WEAVIATE_GRPC_PORT = int(os.getenv("WEAVIATE_GRPC_PORT", "50051"))


def embed_chunks(chunks: list[dict]) -> list[dict]:
    from openai import OpenAI

    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("Thiếu OPENAI_API_KEY trong .env")

    client = OpenAI()
    texts = [chunk["content"] for chunk in chunks]
    embeddings = []
    for start in range(0, len(texts), EMBEDDING_BATCH_SIZE):
        batch = texts[start:start + EMBEDDING_BATCH_SIZE]
        request = {"model": EMBEDDING_MODEL, "input": batch}
        if OPENAI_EMBEDDING_DIM:
            request["dimensions"] = EMBEDDING_DIM
        response = client.embeddings.create(**request)
        embeddings.extend(item.embedding for item in response.data)
        print(f"Embedded {min(start + EMBEDDING_BATCH_SIZE, len(texts))}/{len(texts)} chunks")

    return [{**chunk, "embedding": embedding} for chunk, embedding in zip(chunks, embeddings)]


def connect_weaviate():
    import weaviate
    return weaviate.connect_to_local(
        host=WEAVIATE_HOST,
        port=WEAVIATE_PORT,
        grpc_port=WEAVIATE_GRPC_PORT,
    )


def index_to_vectorstore(embedded_chunks: list[dict]):
    from weaviate.classes.config import Configure, DataType, Property

    client = connect_weaviate()
    try:
        if client.collections.exists(WEAVIATE_COLLECTION):
            client.collections.delete(WEAVIATE_COLLECTION)

        collection = client.collections.create(
            name=WEAVIATE_COLLECTION,
            vector_config=Configure.Vectors.self_provided(),
            properties=[
                Property(name="content", data_type=DataType.TEXT),
                Property(name="source", data_type=DataType.TEXT),
                Property(name="path", data_type=DataType.TEXT),
                Property(name="doc_type", data_type=DataType.TEXT),
                Property(name="chunk_index", data_type=DataType.INT),
            ],
        )

        with collection.batch.dynamic() as batch:
            for chunk in embedded_chunks:
                metadata = chunk["metadata"]
                batch.add_object(
                    properties={
                        "content": chunk["content"],
                        "source": metadata["source"],
                        "path": metadata["path"],
                        "doc_type": metadata["type"],
                        "chunk_index": metadata["chunk_index"],
                    },
                    vector=chunk["embedding"],
                )
    finally:
        client.close()

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Weaviate collection: {WEAVIATE_COLLECTION}")

if RUN_TASK4_INDEX:
    embedded_chunks = embed_chunks(chunks)
    index_to_vectorstore(embedded_chunks)
    print("Indexed chunks to Weaviate.")
else:
    print("Skip indexing. Set RUN_TASK4_INDEX=True to call OpenAI embedding and write Weaviate.")


Embedding model: text-embedding-3-small
Weaviate collection: DrugLawDocs
Skip indexing. Set RUN_TASK4_INDEX=True to call OpenAI embedding and write Weaviate.


In [93]:
# Weaviate Docker helper
# Chạy trong terminal nếu Weaviate chưa bật:
#   docker compose up -d weaviate
# Kiểm tra readiness:
import requests

try:
    ready = requests.get("http://localhost:8080/v1/.well-known/ready", timeout=3)
    print("Weaviate ready status:", ready.status_code, ready.text[:200])
except Exception as exc:
    print("Weaviate chưa sẵn sàng:", exc)

Weaviate ready status: 200 


## Task 5 - Semantic Search

Mục tiêu: embed query bằng cùng OpenAI embedding model và tìm trong Weaviate.

In [94]:
# Task 5 - Semantic Search
# Ưu tiên OpenAI embedding + Weaviate. Nếu lỗi, fallback local bag-of-words để vẫn hiểu flow.

LOCAL_SEMANTIC_CORPUS = []


def embed_query(query: str) -> list[float]:
    from openai import OpenAI
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("Thiếu OPENAI_API_KEY")
    request = {"model": EMBEDDING_MODEL, "input": query}
    if OPENAI_EMBEDDING_DIM:
        request["dimensions"] = EMBEDDING_DIM
    return OpenAI().embeddings.create(**request).data[0].embedding


def tokenize(text: str) -> list[str]:
    return re.findall(r"[\wÀ-ỹ]+", text.lower())


def cosine_from_counts(query_tokens: list[str], doc_tokens: list[str]) -> float:
    q = {}
    d = {}
    for token in query_tokens:
        q[token] = q.get(token, 0) + 1
    for token in doc_tokens:
        d[token] = d.get(token, 0) + 1
    common = set(q) & set(d)
    dot = sum(q[t] * d[t] for t in common)
    q_norm = math.sqrt(sum(v * v for v in q.values()))
    d_norm = math.sqrt(sum(v * v for v in d.values()))
    return dot / (q_norm * d_norm) if q_norm and d_norm else 0.0


def local_semantic_search(query: str, top_k: int = 10) -> list[dict]:
    global LOCAL_SEMANTIC_CORPUS
    if not LOCAL_SEMANTIC_CORPUS:
        LOCAL_SEMANTIC_CORPUS = chunks

    query_tokens = tokenize(query)
    results = []
    for chunk in LOCAL_SEMANTIC_CORPUS:
        score = cosine_from_counts(query_tokens, tokenize(chunk["content"]))
        if score > 0:
            results.append({"content": chunk["content"], "score": float(score), "metadata": chunk["metadata"]})
    return sorted(results, key=lambda item: item["score"], reverse=True)[:top_k]


def semantic_search(query: str, top_k: int = 10) -> list[dict]:
    try:
        from weaviate.classes.query import MetadataQuery

        query_embedding = embed_query(query)
        client = connect_weaviate()
        try:
            if not client.collections.exists(WEAVIATE_COLLECTION):
                return local_semantic_search(query, top_k)
            collection = client.collections.get(WEAVIATE_COLLECTION)
            response = collection.query.near_vector(
                near_vector=query_embedding,
                limit=top_k,
                return_metadata=MetadataQuery(distance=True),
            )
            results = []
            for obj in response.objects:
                props = obj.properties
                distance = obj.metadata.distance
                results.append({
                    "content": props.get("content", ""),
                    "score": float(1 - distance if distance is not None else 0.0),
                    "metadata": {
                        "source": props.get("source"),
                        "path": props.get("path"),
                        "type": props.get("doc_type"),
                        "chunk_index": props.get("chunk_index"),
                    },
                })
            return sorted(results, key=lambda item: item["score"], reverse=True)[:top_k] or local_semantic_search(query, top_k)
        finally:
            client.close()
    except Exception as exc:
        print("Semantic fallback local:", exc)
        return local_semantic_search(query, top_k)

query = "hình phạt cho tội tàng trữ ma túy"
if RUN_SEMANTIC_SEARCH:
    semantic_results = semantic_search(query, top_k=3)
    for r in semantic_results:
        print(f"[{r['score']:.3f}] {r['metadata']}\n{r['content'][:300]}\n")
else:
    print("Skip semantic search.")


[0.367] {'source': 'Luat_Phong_Chong_Ma_Tuy_2021.md', 'path': 'legal/Luat_Phong_Chong_Ma_Tuy_2021.md', 'type': 'legal', 'chunk_index': 17, 'chunking_method': 'recursive'}
chống ma túy.

10. Hướng dẫn sản xuất, hướng dẫn sử dụng trái phép chất ma túy; quảng cáo,

tiếp thị chất ma túy.

11.  Kỳ  thị  người  sử  dụng  trái  phép  chất  ma  túy,  người  cai  nghiện  ma  túy,

người sau cai nghiện ma túy.

12. Các hành vi bị nghiêm cấm khác do luật định liên quan đến ma t

[0.366] {'source': 'Luat_Phong_Chong_Ma_Tuy_2021.md', 'path': 'legal/Luat_Phong_Chong_Ma_Tuy_2021.md', 'type': 'legal', 'chunk_index': 132, 'chunking_method': 'recursive'}
6. Chủ trì thực hiện thống kê nhà nước về phòng, chống ma túy; quản lý thông
tin tội phạm về ma túy, người sử dụng trái phép chất ma túy, người nghiện ma túy,
người  bị  quản  lý  sau  cai  nghiện  ma  túy  và  kết quả  kiểm  soát các  hoạt  động  hợp
pháp liên quan đến ma túy.

7. Thực hiện hợp tác

[0.365] {'source': 'Luat_Phong_Chong_Ma_Tuy_2021.md',

## Task 6 - Lexical Search BM25

Mục tiêu: tìm kiếm keyword bằng BM25 trên corpus markdown/chunks local.

In [95]:
# Task 6 - Lexical Search BM25
# BM25 tốt cho keyword exact match như điều luật, tên người, cụm "tàng trữ trái phép".

BM25_CORPUS = []
BM25_INDEX = None


class SimpleBM25:
    def __init__(self, tokenized_corpus, k1=1.5, b=0.75):
        self.tokenized_corpus = tokenized_corpus
        self.k1 = k1
        self.b = b
        self.doc_count = len(tokenized_corpus)
        self.doc_lengths = [len(doc) for doc in tokenized_corpus]
        self.avg_doc_length = sum(self.doc_lengths) / max(self.doc_count, 1)
        self.term_freqs = []
        self.doc_freqs = {}
        for doc in tokenized_corpus:
            freq = {}
            for token in doc:
                freq[token] = freq.get(token, 0) + 1
            self.term_freqs.append(freq)
            for token in freq:
                self.doc_freqs[token] = self.doc_freqs.get(token, 0) + 1

    def get_scores(self, query_tokens):
        scores = []
        for i, freq in enumerate(self.term_freqs):
            doc_length = self.doc_lengths[i]
            score = 0.0
            for token in query_tokens:
                tf = freq.get(token, 0)
                if tf == 0:
                    continue
                df = self.doc_freqs.get(token, 0)
                idf = math.log(1 + (self.doc_count - df + 0.5) / (df + 0.5))
                denom = tf + self.k1 * (1 - self.b + self.b * doc_length / max(self.avg_doc_length, 1))
                score += idf * (tf * (self.k1 + 1)) / denom
            scores.append(score)
        return scores


def build_bm25_index(corpus):
    tokenized_corpus = [tokenize(doc["content"]) for doc in corpus]
    try:
        from rank_bm25 import BM25Okapi
        return BM25Okapi(tokenized_corpus)
    except ImportError:
        return SimpleBM25(tokenized_corpus)


def lexical_search(query: str, top_k: int = 10) -> list[dict]:
    global BM25_CORPUS, BM25_INDEX
    if not BM25_CORPUS:
        BM25_CORPUS = chunks
    if BM25_INDEX is None:
        BM25_INDEX = build_bm25_index(BM25_CORPUS)

    scores = BM25_INDEX.get_scores(tokenize(query))
    ranked = sorted(range(len(scores)), key=lambda idx: scores[idx], reverse=True)
    results = []
    for idx in ranked[:top_k]:
        score = float(scores[idx])
        if score > 0:
            results.append({"content": BM25_CORPUS[idx]["content"], "score": score, "metadata": BM25_CORPUS[idx]["metadata"]})
    return results


lexical_results = lexical_search("Điều 248 tàng trữ trái phép chất ma túy", top_k=5)
for r in lexical_results:
    print(f"[{r['score']:.3f}] {r['metadata']}\n{r['content'][:300]}\n")


[20.800] {'source': 'article_03.md', 'path': 'news/article_03.md', 'type': 'news', 'chunk_index': 46, 'chunking_method': 'recursive'}
Ca sĩ Sơn Ngọc Minh
Trong đó, 71 bị can bị khởi tố, bắt tạm giam để điều tra về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng trái phép chất ma túy”; 3 trường hợp còn lại bị xử lý hành chính.

[19.873] {'source': 'article_02.md', 'path': 'news/article_02.md', 'type': 'news', 'chunk_index': 15, 'chunking_method': 'recursive'}
Qua công tác điều tra vụ án “Mua bán trái phép chất ma túy, Tàng trữ trái phép chất ma túy và Tổ chức sử dụng trái phép chất ma túy” phát hiện trên địa bàn TPHCM trong quý I/2026, Phòng Cảnh sát điều tra tội phạm về ma túy Công an TPHCM phát hiện ngoài các đối tượng đã bị xử lý, còn nhiều đối tượng 

[18.008] {'source': 'article_02.md', 'path': 'news/article_02.md', 'type': 'news', 'chunk_index': 18, 'chunking_method': 'recursive'}
Kết quả khám phá bước đầu đã bắt giữ, xử lý 74 đối 

## Task 7 - Reranking bằng Qwen

Mục tiêu: rerank candidates từ retrieval. Nếu chưa cài/tải Qwen thì code dùng fallback keyword để notebook vẫn chạy.

In [96]:
# Task 7 - Reranking
# Cross-encoder nếu tải được model; fallback keyword overlap để chạy nhanh khi demo.

QWEN_RERANKER_MODEL = os.getenv("QWEN_RERANKER_MODEL", "Qwen/Qwen3-Reranker-0.6B")
_QWEN_RERANKER = None


def fallback_keyword_rerank(query: str, candidates: list[dict], top_k: int) -> list[dict]:
    query_tokens = set(tokenize(query))
    reranked = []
    for candidate in candidates:
        overlap = len(query_tokens & set(tokenize(candidate.get("content", ""))))
        base_score = float(candidate.get("score", 0.0))
        item = candidate.copy()
        item["original_score"] = base_score
        item["score"] = float(overlap + 0.01 * base_score)
        item["reranker"] = "keyword_fallback"
        reranked.append(item)
    return sorted(reranked, key=lambda item: item["score"], reverse=True)[:top_k]


def rerank(query: str, candidates: list[dict], top_k: int = 5, method: str = "cross_encoder") -> list[dict]:
    global _QWEN_RERANKER
    if not candidates:
        return []
    if method != "cross_encoder":
        return fallback_keyword_rerank(query, candidates, top_k)

    try:
        from sentence_transformers import CrossEncoder
        if _QWEN_RERANKER is None:
            _QWEN_RERANKER = CrossEncoder(QWEN_RERANKER_MODEL, max_length=2048)
        scores = _QWEN_RERANKER.predict(
            [(query, candidate.get("content", "")) for candidate in candidates],
            batch_size=8,
            show_progress_bar=False,
        )
    except Exception as exc:
        print("Rerank fallback local:", exc)
        return fallback_keyword_rerank(query, candidates, top_k)

    reranked = []
    for candidate, score in zip(candidates, scores):
        item = candidate.copy()
        item["original_score"] = float(candidate.get("score", 0.0))
        item["score"] = float(score)
        item["reranker"] = QWEN_RERANKER_MODEL
        reranked.append(item)
    return sorted(reranked, key=lambda item: item["score"], reverse=True)[:top_k]


reranked = rerank("hình phạt ma túy", lexical_results, top_k=3)
for r in reranked:
    print(f"[{r['score']:.3f}] reranker={r.get('reranker')} | original={r.get('original_score')}\n{r['content'][:300]}\n")


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 26874.34it/s]


[1.062] reranker=Qwen/Qwen3-Reranker-0.6B | original=18.00769537208612
Kết quả khám phá bước đầu đã bắt giữ, xử lý 74 đối tượng; trong đó đã khởi tố, bắt tạm giam 71 bị can về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng trái phép chất ma túy”; đồng thời phối hợp Công an phường xử lý hành chính 03 đối tượng theo quy 

[-0.438] reranker=Qwen/Qwen3-Reranker-0.6B | original=17.89682014033658
Ngày 20.5, Công an TP.HCM thông tin đã khởi tố, tạm giam 71 bị can trong một đường dây ma túy lớn trên địa bàn [thành phố](https://thanhnien.vn/thanh-pho.html "thành phố"). Trong số này có ca sĩ Long Nhật và ca sĩ Sơn Ngọc Minh - cựu thành viên nhóm V.Music. Các bị can bị điều tra về các hành vi gồm

[-1.188] reranker=Qwen/Qwen3-Reranker-0.6B | original=20.800215473651747
Ca sĩ Sơn Ngọc Minh
Trong đó, 71 bị can bị khởi tố, bắt tạm giam để điều tra về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng tr

## Task 8 - PageIndex Vectorless Fallback

Mục tiêu: dùng PageIndex nếu có key/SDK; nếu chưa có thì dùng fallback local để xem flow.

In [97]:
# Task 8 - PageIndex Vectorless Fallback
# Nếu chưa có PAGEINDEX_API_KEY, dùng local paragraph overlap để mô phỏng vectorless retrieval.

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "")


def local_vectorless_search(query: str, top_k: int = 5) -> list[dict]:
    query_tokens = set(tokenize(query))
    results = []
    for md_file in sorted(STANDARDIZED_DIR.rglob("*.md")):
        content = md_file.read_text(encoding="utf-8").strip()
        paragraphs = [p.strip() for p in content.split("\n\n") if len(p.strip()) > 80]
        for idx, paragraph in enumerate(paragraphs):
            overlap = len(query_tokens & set(tokenize(paragraph)))
            if overlap == 0:
                continue
            relative_path = md_file.relative_to(STANDARDIZED_DIR)
            results.append({
                "content": paragraph,
                "score": float(overlap),
                "metadata": {
                    "source": md_file.name,
                    "path": str(relative_path),
                    "type": relative_path.parts[0] if relative_path.parts else "unknown",
                    "paragraph_index": idx,
                },
                "source": "pageindex",
            })
    return sorted(results, key=lambda item: item["score"], reverse=True)[:top_k]


def load_pageindex_docs() -> list[dict]:
    if not PAGEINDEX_DOC_IDS_PATH.exists():
        return []
    return json.loads(PAGEINDEX_DOC_IDS_PATH.read_text(encoding="utf-8"))


def save_pageindex_docs(documents: list[dict]):
    PAGEINDEX_DOC_IDS_PATH.write_text(json.dumps(documents, ensure_ascii=False, indent=2), encoding="utf-8")


def upload_documents():
    if not PAGEINDEX_API_KEY:
        raise ValueError("Thiếu PAGEINDEX_API_KEY")
    try:
        from pageindex import PageIndexClient
    except ImportError:
        from pageindex.client import PageIndexClient

    client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
    uploaded = []
    for filepath in sorted(LEGAL_DIR.glob("*.pdf")):
        response = client.submit_document(str(filepath))
        doc_id = response.get("doc_id") or response.get("id") or response.get("document_id")
        uploaded.append({"doc_id": doc_id, "filename": filepath.name, "path": str(filepath.relative_to(PROJECT_ROOT)), "type": "legal"})
        print("Uploaded:", filepath.name, doc_id)
    save_pageindex_docs(uploaded)
    return uploaded


def pageindex_search(query: str, top_k: int = 5) -> list[dict]:
    # Demo-friendly: dùng local fallback nếu chưa cấu hình PageIndex hoặc chưa upload.
    if not PAGEINDEX_API_KEY or not load_pageindex_docs():
        return local_vectorless_search(query, top_k)
    # Có thể mở rộng gọi PageIndex query API ở đây; fallback local vẫn đủ để hiểu flow.
    return local_vectorless_search(query, top_k)


if RUN_PAGEINDEX_UPLOAD:
    uploaded = upload_documents()
    print(f"Uploaded: {len(uploaded)} documents")

pageindex_results = pageindex_search("ma túy nghệ sĩ", top_k=3)
for r in pageindex_results:
    print(f"[{r['score']:.3f}] source={r['source']} {r['metadata']}\n{r['content'][:300]}\n")


[4.000] source=pageindex {'source': 'article_01.md', 'path': 'news/article_01.md', 'type': 'news', 'paragraph_index': 2}
Đóng menu 
[ Chào ngày mới ](https://thanhnien.vn/chao-ngay-moi.htm "Chào ngày mới") [ Tin 24h ](https://thanhnien.vn/tin-24h.htm "Tin 24h") [ Tin thị trường ](https://thanhnien.vn/thi-truong.htm "Tin thị trường") [ Tin 360 ](https://thanhnien.vn/tin-nhanh-360.htm "Tin 360")
[ Video ](https://thanhn

[4.000] source=pageindex {'source': 'article_02.md', 'path': 'news/article_02.md', 'type': 'news', 'paragraph_index': 4}
# Khởi tố, bắt tạm giam ca sĩ Long Nhật, Sơn Ngọc Minh vì tổ chức sử dụng ma túy
##  (Chinhphu.vn) - Ngày 20/5, Cơ quan Cảnh sát điều tra Công an Thành phố Hồ Chí Minh (Phòng Cảnh sát điều tra tội phạm về ma túy) cho biết vừa triệt phá thành công đường dây tội phạm ma tuý quy mô lớn trên địa bàn. 
2

[4.000] source=pageindex {'source': 'article_03.md', 'path': 'news/article_03.md', 'type': 'news', 'paragraph_index': 26}
# Chưa đầy 1 tháng, 3 nghệ sĩ Vi

## Task 9 - Retrieval Pipeline

Mục tiêu: semantic + lexical -> RRF merge -> rerank -> PageIndex fallback nếu score thấp.

In [98]:
# Task 9 - Retrieval Pipeline
# Semantic + lexical -> RRF merge -> optional rerank -> fallback PageIndex nếu score thấp.

SCORE_THRESHOLD = 0.3
DEFAULT_TOP_K = 5


def normalize_result(item: dict, source: str) -> dict:
    item = item.copy()
    item["content"] = item.get("content", "")
    item["score"] = float(item.get("score", 0.0))
    item["metadata"] = item.get("metadata", {}) or {}
    item["source"] = source
    return item


def reciprocal_rank_fusion(ranked_lists: list[list[dict]], top_k: int = 10, k: int = 60) -> list[dict]:
    scores = {}
    content_map = {}
    for ranked_list in ranked_lists:
        for rank, item in enumerate(ranked_list, 1):
            key = item.get("content", "")
            if not key:
                continue
            scores[key] = scores.get(key, 0.0) + 1 / (k + rank)
            content_map[key] = item
    merged = []
    for content, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]:
        item = content_map[content].copy()
        item["score"] = float(score)
        item["reranker"] = "rrf"
        merged.append(item)
    return merged


def retrieve(query: str, top_k: int = DEFAULT_TOP_K, score_threshold: float = SCORE_THRESHOLD, use_reranking: bool = True) -> list[dict]:
    dense_results = semantic_search(query, top_k=top_k * 2)
    sparse_results = lexical_search(query, top_k=top_k * 2)

    merged = reciprocal_rank_fusion([dense_results, sparse_results], top_k=top_k * 3)
    merged = [normalize_result(item, "hybrid") for item in merged]

    if use_reranking and merged:
        final_results = rerank(query, merged, top_k=top_k)
        final_results = [normalize_result(item, "hybrid") for item in final_results]
    else:
        final_results = merged[:top_k]

    best_score = final_results[0]["score"] if final_results else 0.0
    if not final_results or best_score < score_threshold:
        return pageindex_search(query, top_k=top_k)
    return final_results[:top_k]


pipeline_results = retrieve("hình phạt cho tội tàng trữ trái phép chất ma túy", top_k=5)
for r in pipeline_results:
    print(f"[{r['score']:.3f}] source={r['source']} {r.get('metadata', {})}\n{r['content'][:300]}\n")


[5.812] source=hybrid {'source': 'article_02.md', 'path': 'news/article_02.md', 'type': 'news', 'chunk_index': 18, 'chunking_method': 'recursive'}
Kết quả khám phá bước đầu đã bắt giữ, xử lý 74 đối tượng; trong đó đã khởi tố, bắt tạm giam 71 bị can về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng trái phép chất ma túy”; đồng thời phối hợp Công an phường xử lý hành chính 03 đối tượng theo quy 

[5.375] source=hybrid {'source': 'article_04.md', 'path': 'news/article_04.md', 'type': 'news', 'chunk_index': 69, 'chunking_method': 'recursive'}
Theo thông tin mà tôi biết, Công an TP.HCM đã khởi tố, bắt tạm giam 71 bị can trong vụ án liên quan đến các hành vi “mua bán trái phép chất ma túy”, “tàng trữ trái phép chất ma túy” và “tổ chức sử dụng trái phép chất ma túy”, trong đó có ca sĩ Long Nhật và Sơn Ngọc Minh. Trước đó, Công an Hải Phòng 

[5.312] source=hybrid {'source': 'article_04.md', 'path': 'news/article_04.md', 'type': 'news', 'chunk

## Task 10 - Generation có Citation

Mục tiêu: lấy context từ retrieval, reorder để giảm lost-in-the-middle, rồi generate answer có citation. Mặc định cell dưới chỉ preview context; bật `RUN_GENERATION=True` để gọi LLM.

In [99]:
# Task 10 - Generation có Citation
# Retrieve context -> reorder -> format prompt -> gọi LLM hoặc extractive fallback.

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.3
TOP_P = 0.9

SYSTEM_PROMPT = """Answer the question in Vietnamese using only the provided context.
For every factual claim, include a citation in brackets using the source label.
If context is insufficient, say: Tôi không thể xác minh thông tin này từ nguồn hiện có.
"""


def reorder_for_llm(chunks: list[dict]) -> list[dict]:
    if len(chunks) <= 2:
        return chunks
    reordered = []
    for i in range(0, len(chunks), 2):
        reordered.append(chunks[i])
    last_even_index = len(chunks) - 1
    if last_even_index % 2 == 0:
        last_even_index -= 1
    for i in range(last_even_index, 0, -2):
        reordered.append(chunks[i])
    return reordered


def format_context(chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(chunks, 1):
        metadata = chunk.get("metadata", {}) or {}
        source = metadata.get("source", f"Source {i}")
        path = metadata.get("path", "")
        score = float(chunk.get("score", 0.0))
        parts.append(f"[Document {i} | Source: {source} | Path: {path} | Score: {score:.3f}]\n{chunk.get('content', '')}")
    return "\n\n---\n\n".join(parts)


def citation_label(chunk: dict, index: int) -> str:
    metadata = chunk.get("metadata", {}) or {}
    source = metadata.get("source") or f"Document {index}"
    chunk_index = metadata.get("chunk_index")
    return f"{source}, chunk {chunk_index}" if chunk_index is not None else source


def extractive_answer(query: str, chunks: list[dict]) -> str:
    if not chunks:
        return "Tôi không thể xác minh thông tin này từ nguồn hiện có."
    bullets = []
    for i, chunk in enumerate(chunks[:3], 1):
        snippet = " ".join(chunk.get("content", "").split())[:450]
        if snippet:
            bullets.append(f"- {snippet} [{citation_label(chunk, i)}]")
    return "Dựa trên các nguồn đã truy xuất:\n" + "\n".join(bullets) if bullets else "Tôi không thể xác minh thông tin này từ nguồn hiện có."


def generate_with_citation(query: str, top_k: int = 5) -> dict:
    chunks = retrieve(query, top_k=top_k)
    reordered = reorder_for_llm(chunks)
    context = format_context(reordered)
    user_message = f"Context:\n{context}\n\n---\n\nQuestion: {query}"

    try:
        if not os.getenv("OPENAI_API_KEY"):
            raise ValueError("Thiếu OPENAI_API_KEY")
        from openai import OpenAI
        response = OpenAI().chat.completions.create(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )
        answer = response.choices[0].message.content or ""
    except Exception as exc:
        print("LLM fallback extractive:", exc)
        answer = extractive_answer(query, reordered)

    return {
        "answer": answer,
        "sources": chunks,
        "retrieval_source": chunks[0].get("source", "none") if chunks else "none",
    }


question = "Hình phạt cho tội tàng trữ trái phép chất ma túy theo pháp luật Việt Nam?"
context_chunks = reorder_for_llm(pipeline_results)
context = format_context(context_chunks)
print("Reordered context preview:\n")
print(context[:1500])

if RUN_GENERATION:
    result = generate_with_citation(question, top_k=5)
else:
    result = {
        "answer": extractive_answer(question, context_chunks),
        "sources": context_chunks,
        "retrieval_source": context_chunks[0].get("source", "none") if context_chunks else "none",
    }

print("\nAnswer:\n", result["answer"])
print("\nRetrieval source:", result["retrieval_source"])
print("Sources:", len(result["sources"]))


Reordered context preview:

[Document 1 | Source: article_02.md | Path: news/article_02.md | Score: 5.812]
Kết quả khám phá bước đầu đã bắt giữ, xử lý 74 đối tượng; trong đó đã khởi tố, bắt tạm giam 71 bị can về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng trái phép chất ma túy”; đồng thời phối hợp Công an phường xử lý hành chính 03 đối tượng theo quy định pháp luật.

---

[Document 2 | Source: article_04.md | Path: news/article_04.md | Score: 5.312]
Cuối năm 2024, ca sĩ Chi Dân và người mẫu Andrea Aybar (An Tây) bị Công an TP.HCM khởi tố về hành vi tổ chức sử dụng trái phép chất ma túy. Riêng Andrea Aybar còn bị điều tra về hành vi tàng trữ chất cấm. Hồi đầu năm 2025, thông tin NTK Nguyễn Công Trí bị phát hiện sử dụng ma túy tại nhà riêng cũng từng gây xôn xao dư luận. Trước đó, diễn viên Hữu Tín đã bị tuyên phạt 7 năm 6 tháng tù; người mẫu Nhikolai Đinh bị bắt vì tàng trữ ma túy, và ca sĩ Chu Bin bị xử lý hành chính vì sử dụng chất

## Task 11 - RAGAS Evaluation và Report nhóm

Mục tiêu: đọc `group_project/README.md`, load `group_project/evaluation/golden_dataset.json`, chạy RAGAS trên pipeline đã xây trong notebook, rồi ghi/đọc report `results.md`.

Mặc định `RUN_TASK11_RAGAS=False` để không tốn OpenAI calls. Bật `RUN_TASK11_RAGAS=True` ở cell đầu nếu muốn chạy thật.


In [100]:
# Task 11 - Evaluation bằng RAGAS + đọc report trong group_project

GROUP_PROJECT_DIR = PROJECT_ROOT / "group_project"
GROUP_README_PATH = GROUP_PROJECT_DIR / "README.md"
EVALUATION_DIR = GROUP_PROJECT_DIR / "evaluation"
GOLDEN_DATASET_PATH = EVALUATION_DIR / "golden_dataset.json"
RAGAS_RESULTS_PATH = EVALUATION_DIR / "results.md"

print("Group project README:", GROUP_README_PATH)
if GROUP_README_PATH.exists():
    readme_text = GROUP_README_PATH.read_text(encoding="utf-8")
    print(readme_text[:1200])
else:
    print("Không tìm thấy group_project/README.md")


def load_golden_dataset() -> list[dict]:
    return json.loads(GOLDEN_DATASET_PATH.read_text(encoding="utf-8"))


def patch_ragas_vertexai_import():
    """Một số bản RAGAS import optional VertexAI dù mình không dùng."""
    import sys
    import types

    module_name = "langchain_community.chat_models.vertexai"
    if module_name not in sys.modules:
        module = types.ModuleType(module_name)
        module.ChatVertexAI = object
        sys.modules[module_name] = module


def build_ragas_embeddings():
    """answer_relevancy cần object có embed_query/embed_documents."""
    from langchain_openai import OpenAIEmbeddings
    from ragas.embeddings import LangchainEmbeddingsWrapper

    model = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
    return LangchainEmbeddingsWrapper(OpenAIEmbeddings(model=model))


def build_ragas_dataset(golden_dataset: list[dict], limit: int | None = None):
    from datasets import Dataset

    rows = {
        "user_input": [],
        "response": [],
        "retrieved_contexts": [],
        "reference": [],
    }

    items = golden_dataset[:limit] if limit else golden_dataset
    for index, item in enumerate(items, 1):
        question = item["question"]
        print(f"[{index}/{len(items)}] {question}")
        result = generate_with_citation(question, top_k=5)

        rows["user_input"].append(question)
        rows["response"].append(result["answer"])
        rows["retrieved_contexts"].append([source.get("content", "") for source in result.get("sources", [])])
        rows["reference"].append(item["expected_answer"])

    return Dataset.from_dict(rows)


def run_ragas_evaluation(limit: int = 3):
    patch_ragas_vertexai_import()

    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision

    golden_dataset = load_golden_dataset()
    dataset = build_ragas_dataset(golden_dataset, limit=limit)
    metrics = [faithfulness, answer_relevancy, context_recall, context_precision]

    result = evaluate(
        dataset,
        metrics=metrics,
        embeddings=build_ragas_embeddings(),
        raise_exceptions=False,
        show_progress=True,
    )
    df = result.to_pandas()
    metric_names = [metric.name for metric in metrics]

    lines = [
        "# RAG Evaluation Results",
        "",
        f"- Generated at: {datetime.now().isoformat(timespec='seconds')}",
        f"- Evaluated cases: {len(df)}",
        "- Framework: RAGAS",
        "",
        "## Overall Scores",
        "",
        "| Metric | Average Score |",
        "|--------|---------------|",
    ]
    for metric in metric_names:
        if metric in df.columns:
            lines.append(f"| {metric} | {df[metric].mean():.4f} |")

    lines.extend(["", "## Per-Question Scores", ""])
    columns = [column for column in ["user_input", *metric_names] if column in df.columns]
    lines.append(df[columns].to_markdown(index=False))

    RAGAS_RESULTS_PATH.write_text("\n".join(lines), encoding="utf-8")
    print("Saved:", RAGAS_RESULTS_PATH)
    return df


if RUN_TASK11_RAGAS:
    ragas_df = run_ragas_evaluation(limit=RAGAS_EVAL_LIMIT)
    display(ragas_df)
else:
    print("Skip RAGAS. Set RUN_TASK11_RAGAS=True ở cell đầu để chạy evaluation thật.")

# Đọc các report hiện có trong group_project để xem kết quả đã lưu.
report_paths = [
    GROUP_PROJECT_DIR / "result.md",
    GROUP_PROJECT_DIR / "results.md",
    EVALUATION_DIR / "result.md",
    EVALUATION_DIR / "results.md",
]

print("\nExisting reports:")
for path in report_paths:
    if path.exists():
        print("-", path.relative_to(PROJECT_ROOT), f"({path.stat().st_size:,} bytes)")

latest_report = next((path for path in reversed(report_paths) if path.exists()), None)
if latest_report:
    print("\nReport preview:")
    print(latest_report.read_text(encoding="utf-8")[:2000])
else:
    print("Chưa có result.md/results.md trong group_project hoặc group_project/evaluation.")


Group project README: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/group_project/README.md
# Bài Tập Nhóm — Search Engine / RAG Chatbot

## Mục Tiêu

Sau khi hoàn thành bài cá nhân, nhóm ngồi lại để xây dựng **1 trong 2 sản phẩm**:

---

## Yêu cầu 1:  Sản phẩm nhóm RAG Chatbot

Xây dựng chatbot trả lời câu hỏi về pháp luật ma tuý và tin tức liên quan.

**Yêu cầu:**
- Giao diện chat (Streamlit / Gradio / Chainlit)
- Trả lời có citation (dựa trên Task 10)
- Hỗ trợ follow-up questions (conversation memory)
- Hiển thị source documents đã dùng

**Stack gợi ý:**
```
Chainlit/Streamlit → Retrieval (Task 9) → Generation (Task 10) → Display
```

---

## Yêu cầu 2: RAG Evaluation Pipeline

Sử dụng **1 trong 3 framework** sau để evaluate pipeline RAG của nhóm:

### Framework lựa chọn

| Framework | Cài đặt | Đặc điểm |
|-----------|---------|-----------|
| [DeepEval](https://github.com/confident-ai/deepeval) | `pip install deepeval` | Nhiều metric built-in, dễ integrate vớ

## Quick Checklist

- Task 1/2 tạo raw files trong `data/landing/`.
- Task 3 tạo markdown trong `data/standardized/`.
- Task 4 tạo chunks và, nếu bật flag, index vào Weaviate.
- Task 5/6 trả retrieval results.
- Task 7 rerank candidates.
- Task 8 fallback vectorless.
- Task 9 trả kết quả retrieval hợp nhất.
- Task 10 tạo câu trả lời có citation.
- Task 11 chạy RAGAS evaluation và đọc/ghi report trong `group_project/evaluation/`.
